In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
from pathlib import Path
import os
from dotenv import load_dotenv
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from scipy.stats import mannwhitneyu, chi2_contingency, spearmanr

In [ ]:
# ścieżka do aktualnego notebooka / katalogu roboczego
project_root = Path.cwd().parent
print(project_root)
# ścieżka do pliku CSV
data_path = project_root / "data" / "raw" / "BankChurners.csv"

# wczytanie danych
df = pd.read_csv(data_path)

df.head()
#Params for plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
#usunięcie ostatnich dwóch kolumn
df = df.iloc[:, :-2]
#DROP ID column
df = df.drop(columns=['CLIENTNUM'])
df.shape

In [ ]:
df.info()
df.nunique()

In [ ]:
#Checking for missing values
missing_count = df.isnull().sum()
missing_pct = (missing_count / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing_Count': missing_count,
    'Percentage': missing_pct
})
missing_df

In [ ]:
#Check for duplicates
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

In [ ]:
print(f"\nFinal shape: {df.shape[0]} rows and {df.shape[1]} columns")

In [ ]:
#Load data to SQL database
load_dotenv()

engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

df.to_sql("bank_churners", engine, if_exists="replace", index=False)

In [ ]:
#Target variable
target_col = 'Attrition_Flag'
print(f"Target Variable: {target_col}")

# Class distribution
class_dist = df[target_col].value_counts()
class_pct = df[target_col].value_counts(normalize=True) * 100

#Imbalance ratio
imbalance_ratio = class_dist.iloc[1] / class_dist.iloc[0]


print(f"\n Class Distribution:\n{class_dist}")
print(f"\n Class Percentages:\n{class_pct}")
print(f"\nImbalance Ratio: {imbalance_ratio:.2f}")

In [ ]:
plt.figure(figsize = (14,6))
sns.countplot(data=df, x=target_col, hue=target_col)
plt.title('Class Distribution of Attrition_Flag')
plt.show()

In [ ]:
#According to the nunique function some of numerical features are actually ordinal.
#In the model we will treat them as numerical, but for visualization purposes and later for encoding we will separate them.
ordinal_features = [
    'Dependent_count',           
    'Total_Relationship_Count',  
    'Months_Inactive_12_mon',   
    'Contacts_Count_12_mon',
    'Education_Level',
    'Income_Category',
    'Card_Category'
]

numerical_features = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = df.select_dtypes(include=['object', 'str']).columns.tolist()


if target_col in categorical_features:
    categorical_features.remove(target_col)

print("Numerical Features:")
for feat in numerical_features:
    if feat in ordinal_features:
        continue
    print(f" - {feat}")

print(f"\n Categorical Features:")
for feat in categorical_features:
    unique_count = df[feat].nunique()
    print(f" - {feat} ({unique_count} unique values)")
print("\n Ordinal Features:")
for feat in ordinal_features:
    print(f" - {feat}")

In [ ]:
# Detailed statistics for numerical features
print("\n\nDetailed Statistics:")
stats_df = df[numerical_features].describe().T
stats_df['skewness'] = df[numerical_features].skew()
stats_df['kurtosis'] = df[numerical_features].kurtosis()
print(stats_df.round(2))

In [ ]:
#Categorical and ordinal features statistics
print("Categorical features statistics:")
for feat in categorical_features + ordinal_features:
    print(f"\nFeature: {feat}")
    value_counts = df[feat].value_counts()
    percentages = df[feat].value_counts(normalize=True) * 100
    cat_stats_df = pd.DataFrame({
        'Count': value_counts,
        'Percentage': percentages.round(2)
    })
    print(cat_stats_df)

In [ ]:
#Mapping for ordinal features
education_order = [
    'Unknown',
    'Uneducated',
    'High School',
    'College',
    'Graduate',
    'Post-Graduate',
    'Doctorate'
]

income_order = [
    'Unknown',
    'Less than $40K',
    '$40K - $60K',
    '$60K - $80K',
    '$80K - $120K',
    '$120K +'
]

card_order = [
    'Blue',
    'Silver',
    'Gold',
    'Platinum'
]


df['Education_Level'] = pd.Categorical(
    df['Education_Level'],
    categories=education_order,
    ordered=True
)

df['Income_Category'] = pd.Categorical(
    df['Income_Category'],
    categories=income_order,
    ordered=True
)

df['Card_Category'] = pd.Categorical(
    df['Card_Category'],
    categories=card_order,
    ordered=True
)



In [ ]:
for feature in numerical_features:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[feature], kde=True, color='blue', bins=30)
    plt.title(f'Histogram of {feature}')
    plt.xlabel(feature)
    plt.ylabel('Frequency')
    plt.show()


In [ ]:
#Create a countplot categorical feature with different colors for each category
for feature in categorical_features:
    sns.countplot(data = df, x=feature, hue="Attrition_Flag")
    plt.title(f'Count Plot of {feature}')
    plt.xlabel(feature)
    plt.ylabel('Count')
    plt.show()
    

In [ ]:
#BOX PLOTS FOR NUMERICAL FEATURES and potential outliers
for feature in numerical_features:
    if feature in ordinal_features:
        continue
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=df[feature], medianprops={"color": "red"}, boxprops={"facecolor": "lightblue"})
    plt.title(f'Box Plot of {feature}')
    plt.xlabel(feature)
    plt.tight_layout()
    plt.show()

In [ ]:
#Unknown values - we treat them as valid and informative data. 
# Usually in banking industry, "Unknown" can indicate a lack of customer engagement or willingness to 
# share information, which may correlate with churn risk.

In [ ]:
#Outlier detection
outlier_summary = {}
for feature in numerical_features:
    if feature in ordinal_features:
        continue
    Q1 = df[feature].quantile(0.25)
    Q3 = df[feature].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[feature] < lower_bound) | (df[feature] > upper_bound)]
    outlier_count = outliers.shape[0]
    outlier_percentage = (outlier_count / df.shape[0]) * 100
    outlier_summary[feature] = {
        'Outlier_Count': outlier_count,
        'Outlier_Percentage': round(outlier_percentage, 2)
    }

outlier_summary_df = pd.DataFrame(outlier_summary).T
outlier_summary_df
#For banking data, outliers can represent significant customer behaviors, such as unusually high spending or deposits, which are important for churn analysis.
#Therefore, for now we will not remove outliers from the dataset. No feature exceeds 10% of outliers comparing to the total number of records.

In [ ]:
#Bivariate analysis
#Mann Whitney U test for numerical features
significant_numerical = []
for feature in numerical_features:
    if feature in ordinal_features:
        continue
    churned = df[df[target_col] == 'Attrited Customer'][feature]
    retained = df[df[target_col] == 'Existing Customer'][feature]
    stat, p_value = mannwhitneyu(churned, retained)
    if p_value < 0.05:
        significant_numerical.append(feature)
        print(f"Feature '{feature}' is significantly different between classes (p-value: {p_value:.4f})")
    else:
        print(f"Feature '{feature}' is NOT significantly different between classes (p-value: {p_value:.4f}")

In [ ]:
#Chi-Squared test for categorical features
significant_categorical = []
for feature in categorical_features:
    contingency_table = pd.crosstab(df[feature], df[target_col])
    chi2, p_value, dof, expected = chi2_contingency(contingency_table)
    if p_value < 0.05:
        significant_categorical.append(feature)
        print(f"Feature '{feature}' is significantly associated with the target (p-value: {p_value:.4f}, chi2: {chi2:.4f})")
    else:
        print(f"Feature '{feature}' is NOT significantly associated with the target (p-value: {p_value:.4f}, chi2: {chi2:.4f})")

In [ ]:
#Spearman correlation for ordinal features
significant_ordinal = []
target_binary = (df[target_col] == 'Attrited Customer').astype(int)
for feature in ordinal_features:
    corr, p_value = spearmanr(df[feature], target_binary)
    
    if p_value < 0.05:
        significant_ordinal.append(feature)
        direction = "positive" if corr > 0 else "negative"
        print(f"  ✓ {feature}: r={corr:.3f} ({direction}), p={p_value:.4e}")
print(f"\n Significant ordinal features: {significant_ordinal}")

In [ ]:
#Correlation analysis
# Calculate correlation matrix
correlation_matrix = df[numerical_features].corr()
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap='coolwarm', square=True, cbar_kws={"shrink": .8})
plt.title('Correlation Matrix of Numerical Features')
plt.show()